# break detector, inference

Loads the trained 0.879 ensemble and scores new series. Nothing is trained here.

Output: `id,break_probability`

In [ ]:
#settings, change these paths
MODELS = 'cpu_run'                  #folder made by train_087910_cpu.py, it has every saved model
SERIES = 'data/test_series.parquet' #new series, same format as the competition file
OUT = 'predict_run'                 #scratch folder for features, gets created
CSV = 'submission.csv'
WORKERS = 2

In [ ]:
#imports
import os, sys, subprocess
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, os.getcwd())
os.environ['CUDA_VISIBLE_DEVICES'] = ''   #cpu only

In [ ]:
#look at the new data
raw = pd.read_parquet(SERIES)
print(raw.shape, raw['id'].nunique(), 'series')
raw.head()

In [ ]:
#plot a couple of series, the red line is the boundary we score
ids = raw['id'].unique()[:3]
fig, ax = plt.subplots(len(ids), 1, figsize=(10, 6))
for a, i in zip(ax, ids):
    s = raw[raw['id'] == i].sort_values('time')
    a.plot(s['value'].values, lw=.7)
    a.axvline((s['period'] == 0).sum(), ls='--', c='r')
    a.set_title(str(i))
plt.tight_layout()

In [ ]:
#check the saved models are there
for f in ['pipeline/full_cpu_stages.json']:
    assert os.path.exists(f), f
assert os.path.isdir(MODELS), 'train first or point MODELS at the trained folder'
print('ok')

In [ ]:
#run it, this does:
#1 raw series -> features (same formulas as training)
#2 views: reversed, flipped, 4 crops
#3 every saved model predicts, then the fixed readouts combine them
#4 writes id,break_probability
cmd = [sys.executable, '-m', 'pipeline.predict_new', '--models', MODELS, '--series', SERIES,
       '--out', OUT, '--csv', CSV, '--workers', str(WORKERS)]
subprocess.run(cmd, check=True)

In [ ]:
#quick look at the output
sub = pd.read_csv(CSV)
print(sub.shape, sub.break_probability.describe())
sub.head()